# 🚀 [Phase 3] 로컬(RTX 3060) 경량 모델 풀학습
이 노트북은 가벼운 CNN 모델들(ResNet50, ReDimNet2, ECAPA_TDNN, CAMPP)을 로컬 PC에서 100% 데이터로 훈련하기 위한 깔끔한 전용 노트북입니다.

In [1]:
import torch
import sys, os

print(f"PyTorch 버전: {torch.__version__}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU 활성화 성공: {gpu_name} (총 VRAM: {vram_gb:.1f} GB)")
else:
    print("⚠️ GPU 가속기가 활성화되지 않았습니다!")


PyTorch 버전: 2.7.1+cu118
✅ GPU 활성화 성공: NVIDIA GeForce RTX 3060 Laptop GPU (총 VRAM: 6.0 GB)


In [2]:
import os
import torch
import multiprocessing
from benchmark_suite.dataset import UniversalSpeakerDataset
from benchmark_suite.models import build_model
from benchmark_suite.trainer import BenchmarkTrainer
from benchmark_suite.config import MODEL_REGISTRY
from torch.utils.data import DataLoader

# 경로 설정
DATA_ROOT = r"C:\Users\user\Desktop\DCC\data"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR = os.path.join(DATA_ROOT, "val")
OUTPUT_DIR = "./test_results"

print("✅ 경로 세팅 완료!")


✅ 경로 세팅 완료!


In [ ]:
# 트레이너 초기화 (드라이브 백업 경로도 로컬로 지정)
trainer = BenchmarkTrainer(
    output_dir=OUTPUT_DIR,
    drive_backup_dir=OUTPUT_DIR
)

PHASE2_MODELS = ["resnet50", "redimnet", "ecapa_tdnn", "campp"]

for model_name in PHASE2_MODELS:
    cfg = MODEL_REGISTRY.get(model_name, {})
    
    train_dataset = UniversalSpeakerDataset(TRAIN_DIR, model_name=model_name, config_dict=cfg, is_train=True)
    val_dataset = UniversalSpeakerDataset(VAL_DIR, model_name=model_name, config_dict=cfg, is_train=False)
    
    # 로컬 메모리를 고려하여 batch_size 16으로 안전하게 설정
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0, pin_memory=True)
    
    model = build_model(model_name, num_classes=2, pretrained=True)
    
    # 이어학습(Auto-Resume) 기능이 켜져 있으므로, 90% 가중치가 있으면 자동으로 불러옵니다!
    trainer.fit(
        model_name=model_name,
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=15,
        lr=1e-4,
        skip_if_done=False
    )

print("✅ 로컬 학습이 모두 완료되었습니다!")



🚀 [모델 벤치마크 실행] : RESNET50
🔄 이전 체크포인트를 발견했습니다! [./test_results\checkpoints\best_resnet50.pt] 불러와서 이어서 학습(Fine-Tuning)합니다.
⚠️ 구글 드라이브 동기화 일시 오류: './test_results\\checkpoints\\step_resnet50.pt' and './test_results\\checkpoints\\step_resnet50.pt' are the same file
⚠️ 구글 드라이브 동기화 일시 오류: './test_results\\checkpoints\\step_resnet50.pt' and './test_results\\checkpoints\\step_resnet50.pt' are the same file
⚠️ 구글 드라이브 동기화 일시 오류: './test_results\\checkpoints\\step_resnet50.pt' and './test_results\\checkpoints\\step_resnet50.pt' are the same file
⚠️ 구글 드라이브 동기화 일시 오류: './test_results\\checkpoints\\step_resnet50.pt' and './test_results\\checkpoints\\step_resnet50.pt' are the same file
⚠️ 구글 드라이브 동기화 일시 오류: './test_results\\checkpoints\\step_resnet50.pt' and './test_results\\checkpoints\\step_resnet50.pt' are the same file
⚠️ 구글 드라이브 동기화 일시 오류: './test_results\\checkpoints\\step_resnet50.pt' and './test_results\\checkpoints\\step_resnet50.pt' are the same file
⚠️ 구글 드라이브 동기화 일시 오류: './test_results\